In [0]:
%sql
-- 1. Truncate tables for a clean slate
TRUNCATE TABLE analytics_dq_dev.configuration.run_policy;
TRUNCATE TABLE analytics_dq_dev.configuration.rule_template;
TRUNCATE TABLE analytics_dq_dev.configuration.rule_assignment;

-- 2. Create test_trips (DIRTY table - we will corrupt this one with Errors)
CREATE OR REPLACE TABLE analytics_dq_dev.metadata.test_trips AS 
SELECT * FROM samples.nyctaxi.trips;

-- 3. Create test_trips_clean (CLEAN table - we will leave this one pristine)
CREATE OR REPLACE TABLE analytics_dq_dev.metadata.test_trips_clean AS 
SELECT * FROM samples.nyctaxi.trips;

-- 4. Create test_trips_warning (WARNING table - we will only trigger Warnings here)
CREATE OR REPLACE TABLE analytics_dq_dev.metadata.test_trips_warning AS 
SELECT * FROM samples.nyctaxi.trips;


---------------------------------------------------------------------
-- CORRUPTING THE `test_trips` TABLE TO DEMONSTRATE ERRORS
---------------------------------------------------------------------

-- 5. Case A: Single Nulls on fare_amount
-- Set pickup_zip = 99999 to guarantee they fall into the MAX partition evaluated by filter_field
UPDATE analytics_dq_dev.metadata.test_trips 
SET fare_amount = NULL, pickup_zip = 99999 
WHERE tpep_pickup_datetime IN (
    SELECT tpep_pickup_datetime FROM analytics_dq_dev.metadata.test_trips WHERE trip_distance > 10 LIMIT 3
);

-- 6. Case B: Entire column NULL on dropoff_zip (No quarantine intended)
UPDATE analytics_dq_dev.metadata.test_trips 
SET dropoff_zip = NULL;

-- 7. Case C: Out of bounds (Dynamic thresholds) on trip_distance
UPDATE analytics_dq_dev.metadata.test_trips 
SET trip_distance = -5.0, pickup_zip = 99999 
WHERE tpep_pickup_datetime IN (
    SELECT tpep_pickup_datetime FROM analytics_dq_dev.metadata.test_trips WHERE fare_amount > 5 LIMIT 1
);

UPDATE analytics_dq_dev.metadata.test_trips 
SET trip_distance = 250.0, pickup_zip = 99999 
WHERE tpep_pickup_datetime IN (
    SELECT tpep_pickup_datetime FROM analytics_dq_dev.metadata.test_trips WHERE fare_amount > 10 LIMIT 1
);

-- 8. Case D: Duplicate Composite Primary Key (tpep_pickup_datetime, pickup_zip)
-- Prepare an existing record by setting pickup_zip = 99999
UPDATE analytics_dq_dev.metadata.test_trips 
SET pickup_zip = 99999
WHERE tpep_pickup_datetime IN (
    SELECT tpep_pickup_datetime FROM analytics_dq_dev.metadata.test_trips WHERE fare_amount > 8 LIMIT 1
);
-- and then INSERT an exact copy of it to create a duplicate in the 99999 partition!
INSERT INTO analytics_dq_dev.metadata.test_trips 
SELECT * FROM analytics_dq_dev.metadata.test_trips 
WHERE pickup_zip = 99999 AND fare_amount > 8 LIMIT 1;


---------------------------------------------------------------------
-- CORRUPTING THE `test_trips_warning` TABLE TO DEMONSTRATE WARNINGS
---------------------------------------------------------------------

-- 9. Case E: Warning for test_trips_warning
-- Make some dropoff_zip NULLs to trigger the Warning rule on the Warning table
UPDATE analytics_dq_dev.metadata.test_trips_warning 
SET dropoff_zip = NULL, pickup_zip = 99999 
WHERE tpep_pickup_datetime IN (
    SELECT tpep_pickup_datetime FROM analytics_dq_dev.metadata.test_trips_warning WHERE trip_distance > 5 LIMIT 4
);